In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_err):
    # only convert if 'timestamp' column exists
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Report overall time range and unique cmdb_id for each file
metric_time_range = (df_metric['timestamp'].min(), df_metric['timestamp'].max()) if not df_metric.empty else (None, None)
metric_unique_cmdb = sorted(df_metric['cmdb_id'].unique())

trace_time_range = (df_trace['timestamp'].min(), df_trace['timestamp'].max()) if not df_trace.empty else (None, None)
trace_unique_cmdb = sorted(df_trace['cmdb_id'].unique())

log_time_range = (df_log['timestamp'].min(), df_log['timestamp'].max()) if not df_log.empty else (None, None)
log_unique_cmdb = sorted(df_log['cmdb_id'].unique())

err_time_range = (df_err['timestamp'].min(), df_err['timestamp'].max()) if not df_err.empty else (None, None)
err_unique_cmdb = sorted(df_err['cmdb_id'].unique())

# 2) metric.csv aggregation by (cmdb_id, kpi_name)
if not df_metric.empty:
    grp_metric = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    metric_agg = grp_metric.agg(count='count', mean='mean', min='min', max='max').reset_index()
    # global P95 per group
    metric_p95 = grp_metric.quantile(0.95).rename('p95').reset_index()
    metric_agg = metric_agg.merge(metric_p95, on=['cmdb_id','kpi_name'])
    # rounding for compactness
    metric_agg[['mean','min','max','p95']] = metric_agg[['mean','min','max','p95']].round(6)
    metric_agg_top50 = metric_agg.sort_values('count', ascending=False).head(50)
else:
    metric_agg_top50 = pd.DataFrame(columns=['cmdb_id','kpi_name','count','mean','min','max','p95'])

# 3) trace.csv aggregation by (cmdb_id, trace_name)
if not df_trace.empty:
    grp_trace = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
    trace_agg = grp_trace.agg(count='count', mean='mean', min='min', max='max').reset_index()
    trace_p95 = grp_trace.quantile(0.95).rename('p95').reset_index()
    trace_agg = trace_agg.merge(trace_p95, on=['cmdb_id','trace_name'])
    trace_agg[['mean','min','max','p95']] = trace_agg[['mean','min','max','p95']].round(6)
    trace_agg_top50 = trace_agg.sort_values('count', ascending=False).head(50)
else:
    trace_agg_top50 = pd.DataFrame(columns=['cmdb_id','trace_name','count','mean','min','max','p95'])

# 4) log.csv aggregation by (cmdb_id, log_name)
if not df_log.empty:
    grp_log = df_log.groupby(['cmdb_id', 'log_name'])['value']
    log_agg = grp_log.agg(count='count', mean='mean', min='min', max='max').reset_index()
    log_p95 = grp_log.quantile(0.95).rename('p95').reset_index()
    log_agg = log_agg.merge(log_p95, on=['cmdb_id','log_name'])
    log_agg[['mean','min','max','p95']] = log_agg[['mean','min','max','p95']].round(6)
    log_agg_top50 = log_agg.sort_values('count', ascending=False).head(50)
else:
    log_agg_top50 = pd.DataFrame(columns=['cmdb_id','log_name','count','mean','min','max','p95'])

# 5) error_logs summary: count, unique cmdb_id, earliest 20 rows
err_count = len(df_err)
err_unique_cmdb = sorted(df_err['cmdb_id'].unique())
if err_count > 0:
    err_preview = df_err.sort_values('timestamp')[['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)
else:
    err_preview = pd.DataFrame(columns=['timestamp','cmdb_id','message'])

# Keep final displayed output compact: only aggregates and small summaries
metric_time_range, metric_unique_cmdb, metric_agg_top50, trace_time_range, trace_unique_cmdb, trace_agg_top50, log_time_range, log_unique_cmdb, log_agg_top50, (err_count, err_unique_cmdb, err_preview)

```
Out[1]:
```
```python
# Compose a concise plain-English summary using previously computed variables
# (assumes metric_time_range, trace_time_range, log_time_range, err_count,
#  metric_unique_cmdb, trace_unique_cmdb, log_unique_cmdb,
#  metric_agg_top50, trace_agg_top50, log_agg_top50 are available in the kernel)

def safe_get_metric(cmdb, kpi, col='p95'):
    try:
        row = metric_agg_top50.query("cmdb_id == @cmdb and kpi_name == @kpi")
        if row.empty:
            return "n/a"
        val = float(row.iloc[0][col])
        return f"{val:.6g}"
    except Exception:
        return "n/a"

def safe_get_trace(cmdb, trace_name, col='mean'):
    try:
        row = trace_agg_top50.query("cmdb_id == @cmdb and trace_name == @trace_name")
        if row.empty:
            return "n/a"
        val = float(row.iloc[0][col])
        return f"{val:.6g}"
    except Exception:
        return "n/a"

def safe_get_log(cmdb, logname, col='mean'):
    try:
        row = log_agg_top50.query("cmdb_id == @cmdb and log_name == @logname")
        if row.empty:
            return "n/a"
        val = float(row.iloc[0][col])
        return f"{val:.6g}"
    except Exception:
        return "n/a"

# Key metric highlights (safe lookups)
ad_diskio_p95 = safe_get_metric('adservice', 'diskio', 'p95')
ad_mem_mean = safe_get_metric('adservice', 'mem', 'mean')
product_cpu_mean = safe_get_metric('productcatalogservice', 'cpu', 'mean')
payment_socket_p95 = safe_get_metric('paymentservice', 'socket', 'p95')

# Key trace highlights
checkout_rowcount_mean = safe_get_trace('checkoutservice', 'trace.from_root.row_count', 'mean')
checkout_duration_p95 = safe_get_trace('checkoutservice', 'trace.from_root.duration_p95', 'p95')
frontend_to_product_rowcount_mean = safe_get_trace('frontendservice', 'trace.to_productcatalogservice.row_count', 'mean')

# Key log highlights
frontend_rowcount_mean = safe_get_log('frontend', 'log.row_count', 'mean')
currency_rowcount_mean = safe_get_log('currencyservice', 'log.row_count', 'mean')

summary = (
    "Summary of telemetry (plain English):\n\n"
    f"- Time ranges (UTC):\n"
    f"  • metric: {metric_time_range[0]} to {metric_time_range[1]}\n"
    f"  • trace:  {trace_time_range[0]} to {trace_time_range[1]}\n"
    f"  • log:    {log_time_range[0]} to {log_time_range[1]}\n\n"
    f"- Unique services observed:\n"
    f"  • metric cmdb_ids: {metric_unique_cmdb}\n"
    f"  • trace cmdb_ids:  {trace_unique_cmdb}\n"
    f"  • log cmdb_ids:    {log_unique_cmdb}\n\n"
    "- Key metric observations (top groups by sample count):\n"
    f"  • adservice disk I/O p95 ≈ {ad_diskio_p95}\n"
    f"  • adservice memory mean ≈ {ad_mem_mean}\n"
    f"  • productcatalogservice CPU mean ≈ {product_cpu_mean}\n"
    f"  • paymentservice socket values appear constant; p95 ≈ {payment_socket_p95}\n\n"
    "- Key trace observations:\n"
    f"  • checkoutservice from_root row_count mean ≈ {checkout_rowcount_mean}, duration_p95 ≈ {checkout_duration_p95}\n"
    f"  • frontend → productcatalogservice row_count mean ≈ {frontend_to_product_rowcount_mean}\n"
    f"  • Many trace error_rate entries are 0 (no trace-level errors observed in aggregates).\n\n"
    "- Key log observations:\n"
    f"  • frontend log.row_count mean ≈ {frontend_rowcount_mean}\n"
    f"  • currencyservice log.row_count mean ≈ {currency_rowcount_mean}\n"
    f"  • log.error_count is 0 for most services in the aggregates (no widespread log error spikes).\n\n"
    f"- error_logs.csv: row count = {err_count}. No error_log rows were returned (empty preview).\n\n"
    "Overall conclusion: telemetry covers the same short window (2024-01-19 00:06–00:30 UTC). "
    "Aggregates show normal/expected row counts across services, low error_count and trace error_rate = 0. "
    "Notable numeric highlights are high adservice disk I/O and service-specific CPU/memory magnitudes, "
    "but there are no obvious widespread error spikes in logs/traces within this window."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

((Timestamp('2024-01-19 00:06:00+0000', tz='UTC'), Timestamp('2024-01-19 00:30:00+0000', tz='UTC')), ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'frontend-external', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis', 'shippingservice'],                   cmdb_id    kpi_name  count          mean           min           max           p95
0               adservice         cpu     25  7.882430e-01  6.016320e-01  2.406246e+00  1.054467e+00
1               adservice      diskio     25  6.227988e+04  3.704578e+04  7.715921e+04  7.494087e+04
2               adservice  latency-50     25  3.006000e-03  2.987000e-03  3.057000e-03  3.026000e-03
3               adservice  latency-90     25  4.626000e-03  4.604000e-03  4.707000e-03  4.660000e-03
4               adservice         mem     25  1.130780e+08  1.124393e+08  1.187755e+08  1.153011e+08
..                    ...         ...    ...           ...           ...           ...           ...
45         paymentservice  latency-90     25  8.587000e-03  7.599000e-03  9.621000e-03  9.308000e-03
46         paymentservice         mem     25  4.037429e+07  3.981339e+07  4.114528e+07  4.109697e+07
47         paymentservice      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00
48         paymentservice    workload     25  8.779760e-01  7.121500e-01  1.063267e+00  1.000583e+00
49  productcatalogservice         cpu     25  4.252063e+00  3.998856e+00  4.595880e+00  4.523790e+00

[50 rows x 7 columns], (Timestamp('2024-01-19 00:06:00+0000', tz='UTC'), Timestamp('2024-01-19 00:30:00+0000', tz='UTC')), ['checkoutservice', 'currencyservice', 'emailservice', 'frontendservice', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'root'],             cmdb_id                                trace_name  count         mean        min          max          p95
11  checkoutservice                 trace.from_root.row_count     25   112.200000   1.000000   137.000000   135.000000
10  checkoutservice                trace.from_root.error_rate     25     0.000000   0.000000     0.000000     0.000000
9   checkoutservice              trace.from_root.duration_p95     25     0.071596   0.002977     0.089270     0.087901
8   checkoutservice             trace.from_root.duration_mean     25     0.014420   0.002977     0.024183     0.020612
75  frontendservice  trace.to_productcatalogservice.row_count     25  3520.800000  36.000000  3933.000000  3898.400000
..              ...                                       ...    ...          ...        ...          ...          ...
70  frontendservice       trace.to_frontendservice.error_rate     25     0.000000   0.000000     0.000000     0.000000
69  frontendservice     trace.to_frontendservice.duration_p95     25     0.013431   0.007206     0.020102     0.019022
68  frontendservice    trace.to_frontendservice.duration_mean     25     0.006294   0.003684     0.008007     0.007739
64  frontendservice    trace.to_currencyservice.duration_mean     25     0.000296   0.000146     0.000457     0.000418
65  frontendservice     trace.to_currencyservice.duration_p95     25     0.000217   0.000206     0.000235     0.000232

[50 rows x 7 columns], (Timestamp('2024-01-19 00:06:00+0000', tz='UTC'), Timestamp('2024-01-19 00:30:00+0000', tz='UTC')), ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'paymentservice', 'recommendationservice', 'redis', 'shippingservice'],                   cmdb_id         log_name  count         mean  min   max     p95
0               adservice  log.error_count     25     0.000000    0     0     0.0
1               adservice    log.row_count     25   435.240000    4   473   466.8
2             cartservice  log.error_count     25     0.000000    0     0     0.0
3             cartservice    log.row_count     25   738.800000    7   827   804.2
7         currencyservice    log.row_count     25  2285.320000   26  2554  2522.2
6         currencyservice  log.error_count     25     0.000000    0     0     0.0
10               frontend  log.error_count     25     0.000000    0     0     0.0
11               frontend    log.row_count     25  2561.880000   27  2791  2761.2
19        shippingservice    log.row_count     25   423.840000    4   512   485.6
18        shippingservice  log.error_count     25     0.000000    0     0     0.0
14  recommendationservice  log.error_count     25     0.000000    0     0     0.0
15  recommendationservice    log.row_count     25   540.680000    5   611   601.6
8            emailservice  log.error_count     24     0.000000    0     0     0.0
9            emailservice    log.row_count     24    26.333333   20    33    32.0
5         checkoutservice    log.row_count     24    79.000000   60    99    96.0
4         checkoutservice  log.error_count     24     0.000000    0     0     0.0
12         paymentservice  log.error_count     24     0.000000    0     0     0.0
13         paymentservice    log.row_count     24    52.666667   40    66    64.0
17                  redis    log.row_count      5     5.000000    5     5     5.0
16                  redis  log.error_count      5     0.000000    0     0     0.0, (0, [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: []))

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse existing df_metric (assumed loaded and timestamp parsed). If not present, load it.
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric series (before any time filtering)
p95_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# Candidate services and incident window (UTC)
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']
start = pd.Timestamp('2024-01-19 00:03:59', tz='UTC')
end = pd.Timestamp('2024-01-19 00:33:59', tz='UTC')

# 2) Filter metric rows to incident window and candidate services
df_window = df_metric[
    (df_metric['timestamp'] >= start) &
    (df_metric['timestamp'] <= end) &
    (df_metric['cmdb_id'].isin(candidates))
].copy()

total_rows_in_window = len(df_window)

# 3) Merge windowed rows with global p95 thresholds
df_window = df_window.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Find anomaly rows where value >= global_p95
# Note: If global_p95 is NaN (no global p95), these will be excluded
df_anoms = df_window[df_window['value'] >= df_window['global_p95']].copy()

# 5) Aggregate anomalies per (cmdb_id, kpi_name)
if not df_anoms.empty:
    agg = (
        df_anoms
        .groupby(['cmdb_id', 'kpi_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_anomaly_value=('value', 'max'),
            global_p95=('global_p95', 'first')
        )
    )
    # compute severity_ratio with safe division
    agg['severity_ratio'] = agg['max_anomaly_value'] / agg['global_p95']
    # Round numeric columns for compactness
    agg['global_p95'] = agg['global_p95'].round(6)
    agg['max_anomaly_value'] = agg['max_anomaly_value'].round(6)
    agg['severity_ratio'] = agg['severity_ratio'].replace([np.inf, -np.inf], np.nan).round(6)
    # Keep only anomaly_count > 0 (should be true) and sort
    metric_anom_summary = agg[agg['anomaly_count'] > 0].sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    metric_anom_summary = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','global_p95','severity_ratio'])

# Final compact outputs
metric_anom_summary, total_rows_in_window

```
Out[2]:
```
Summary of metric anomalies for the candidate services in the incident window (2024-01-19 00:03:59 – 00:33:59 UTC):

- Total metric rows for the candidate services in the window: 900.

- Overall: many groups report rows >= their global P95. However several groups (diskio, error) have global_p95 = 0 and max = 0, producing anomaly_count = 25 but meaningless severity (NaN). Also several socket groups equal their global_p95 (severity ≈ 1.0), indicating constant socket values that meet but do not greatly exceed thresholds.

- Most notable meaningful anomalies (severity > 1 and non-zero thresholds):
  - emailservice — mem: anomaly_count = 3, earliest = 2024-01-19 00:23:00 UTC, max = 43,460,200, global_p95 ≈ 43,438,080, severity ≈ 1.00051 (small exceedances).
  - checkoutservice — cpu: anomaly_count = 2, earliest = 2024-01-19 00:25:00 UTC, max ≈ 19.99317, global_p95 ≈ 19.83767, severity ≈ 1.00784.
  - checkoutservice — latency-90: anomaly_count = 2, earliest = 2024-01-19 00:26:00 UTC, max ≈ 0.475301, global_p95 ≈ 0.459748, severity ≈ 1.03383.
  - checkoutservice — latency-50: anomaly_count = 2, earliest = 2024-01-19 00:29:00 UTC, severity ≈ 1.00939.
  - checkoutservice — mem: anomaly_count = 2, earliest = 2024-01-19 00:24:00 UTC, severity ≈ 1.00333.
  - checkoutservice — workload: anomaly_count = 2, earliest = 2024-01-19 00:09:00 UTC, severity ≈ 1.08780.
  - currencyservice — latency-90 and latency-50: each anomaly_count = 2 (earliest 00:13 and 00:24 UTC respectively), severity for latency-90 ≈ 1.10898, for latency-50 ≈ 1.00047.
  - currencyservice — mem: anomaly_count = 2, earliest = 00:07:00 UTC, severity ≈ 1.00353.

- Items to treat with caution:
  - diskio and error groups across several services show anomaly_count = 25 but global_p95 = 0 and max = 0 (these are not actionable as severity is undefined).
  - Several socket groups (productcatalogservice, recommendationservice, currencyservice, paymentservice, emailservice) show anomaly_count = 25 with max == global_p95 and severity = 1.0 (constant values at the threshold).

Bottom line: meaningful exceedances are small in magnitude and limited to a few metrics/services, with checkoutservice and currencyservice showing the clearest latency/CPU/memory exceedances (small severity ratios >1). Disk I/O and some error metrics flagged many "anomalies" only because their global P95 is zero and should be interpreted cautiously.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name  anomaly_count earliest_anomaly_timestamp  max_anomaly_value    global_p95  severity_ratio
0         checkoutservice      diskio             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
1            emailservice      diskio             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
2            emailservice      socket             25  2024-01-19 00:06:00+00:00       3.000000e+00  3.000000e+00        1.000000
3         currencyservice      diskio             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
4   recommendationservice      socket             25  2024-01-19 00:06:00+00:00       4.000000e+00  4.000000e+00        1.000000
5   productcatalogservice      socket             25  2024-01-19 00:06:00+00:00       4.000000e+00  4.000000e+00        1.000000
6   productcatalogservice      diskio             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
7         currencyservice      socket             25  2024-01-19 00:06:00+00:00       4.000000e+00  4.000000e+00        1.000000
8   recommendationservice      diskio             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
9   productcatalogservice       error             25  2024-01-19 00:06:00+00:00       0.000000e+00  0.000000e+00             NaN
10           emailservice         mem              3  2024-01-19 00:23:00+00:00       4.346020e+07  4.343808e+07        1.000509
11        checkoutservice         cpu              2  2024-01-19 00:25:00+00:00       1.999317e+01  1.983767e+01        1.007839
12        checkoutservice  latency-90              2  2024-01-19 00:26:00+00:00       4.753010e-01  4.597480e-01        1.033830
13        checkoutservice  latency-50              2  2024-01-19 00:29:00+00:00       2.826850e-01  2.800560e-01        1.009387
14        currencyservice  latency-50              2  2024-01-19 00:24:00+00:00       3.669000e-03  3.668000e-03        1.000474
15        currencyservice  latency-90              2  2024-01-19 00:13:00+00:00       3.564200e-02  3.214000e-02        1.108978
16        currencyservice         mem              2  2024-01-19 00:07:00+00:00       4.735816e+07  4.719164e+07        1.003529
17        checkoutservice         mem              2  2024-01-19 00:24:00+00:00       1.895096e+07  1.888805e+07        1.003331
18        checkoutservice    workload              2  2024-01-19 00:09:00+00:00       1.084483e+00  9.969500e-01        1.087801
19        checkoutservice      socket              2  2024-01-19 00:26:00+00:00       1.130000e+01  1.113333e+01        1.014970, 900)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Ensure df_trace is loaded and timestamp parsed (reuse if present)
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure timestamp dtype is datetime64[ns, UTC]
if not pd.api.types.is_datetime64_any_dtype(df_trace['timestamp']):
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) using the full trace series
trace_p95 = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)

# 2) Define incident window and candidate services
start = pd.Timestamp('2024-01-19 00:03:59', tz='UTC')
end = pd.Timestamp('2024-01-19 00:33:59', tz='UTC')
candidates = ['checkoutservice','currencyservice','emailservice']

# 3) Filter trace rows to incident window and candidate services
df_trace_window = df_trace[
    (df_trace['timestamp'] >= start) &
    (df_trace['timestamp'] <= end) &
    (df_trace['cmdb_id'].isin(candidates))
].copy()

total_trace_rows_in_window = len(df_trace_window)

# 4) Merge with global P95 thresholds (computed from full series)
df_trace_window = df_trace_window.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# 5) Identify anomaly rows where value >= global_p95 (exclude null thresholds)
df_trace_anoms = df_trace_window[
    df_trace_window['global_p95'].notna() &
    (df_trace_window['value'] >= df_trace_window['global_p95'])
].copy()

# 6) Aggregate anomalies per (cmdb_id, trace_name)
if not df_trace_anoms.empty:
    trace_agg = (
        df_trace_anoms
        .groupby(['cmdb_id', 'trace_name'], as_index=False)
        .agg(
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_anomaly_value=('value', 'max'),
            global_p95=('global_p95', 'first')
        )
    )
    # compute severity_ratio safely; if global_p95 == 0 -> NaN
    trace_agg['severity_ratio'] = np.where(
        trace_agg['global_p95'] == 0,
        np.nan,
        trace_agg['max_anomaly_value'] / trace_agg['global_p95']
    )
    # Round numeric columns for compactness
    trace_agg['global_p95'] = trace_agg['global_p95'].round(6)
    trace_agg['max_anomaly_value'] = trace_agg['max_anomaly_value'].round(6)
    trace_agg['severity_ratio'] = pd.Series(trace_agg['severity_ratio']).replace([np.inf, -np.inf], np.nan).round(6)
    # Keep only rows with anomaly_count > 0 (ensured) and sort/limit
    trace_anom_summary = trace_agg.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
    # Ensure column order
    trace_anom_summary = trace_anom_summary[['cmdb_id','trace_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio']]
else:
    trace_anom_summary = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','anomaly_count','earliest_anomaly_timestamp','max_anomaly_value','severity_ratio'])

# Final compact outputs
trace_anom_summary, total_trace_rows_in_window

```
Out[3]:
```
Summary of trace anomalies for the candidate services in the incident window (2024-01-19 00:03:59 – 00:33:59 UTC):

- Total trace rows for the candidate services in the window: 1260.

- General note: many flagged "anomalies" are for trace error_rate groups whose global P95 = 0 and values = 0. Those are not actionable (severity undefined). Treat them as non-problematic.

- Meaningful anomalies (non-zero thresholds / severity > 1):
  - checkoutservice — trace.from_root.row_count: anomaly_count = 3, earliest = 2024-01-19 00:07:00 UTC, max = 137, global_p95 = 135, severity ≈ 1.015 (small exceedance).
  - checkoutservice — several row_count metrics (to_paymentservice.row_count, from_frontendservice.row_count, to_currencyservice.row_count): anomaly_count = 3 each, earliest ≈ 00:11 or 00:07 UTC, max values slightly above p95, severity ≈ 1.018–1.031 (small increases).
  - emailservice — trace.from_checkoutservice.row_count and trace.from_checkoutservice.row_count variants: anomaly_count = 3, earliest ≈ 00:11 UTC, max slightly above p95, severity ≈ 1.031.
  - currencyservice — trace.from_root.row_count: anomaly_count = 24, but max == global_p95 (severity = 1.0) — no real exceedance beyond threshold.

- Bottom line: there are no large or sustained trace error spikes. The only notable findings are small, short-lived increases in row_count for checkoutservice (and small row_count bumps for emailservice/currencyservice), and many error_rate entries are zero and not actionable. Overall, checkoutservice has the most meaningful trace exceedances but magnitudes are small.

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                 trace_name  global_p95  anomaly_count earliest_anomaly_timestamp  max_anomaly_value  severity_ratio
0   checkoutservice                 trace.from_root.error_rate         0.0             25  2024-01-19 00:06:00+00:00                0.0             NaN
1      emailservice                 trace.from_root.error_rate         0.0             25  2024-01-19 00:06:00+00:00                0.0             NaN
2   currencyservice      trace.from_frontendservice.error_rate         0.0             25  2024-01-19 00:06:00+00:00                0.0             NaN
3   checkoutservice      trace.from_checkoutservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
4   checkoutservice        trace.to_checkoutservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
5   currencyservice      trace.from_checkoutservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
6   currencyservice                 trace.from_root.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
7   currencyservice                  trace.from_root.row_count        12.0             24  2024-01-19 00:07:00+00:00               12.0        1.000000
8   checkoutservice         trace.to_paymentservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
9   checkoutservice           trace.to_emailservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
10  checkoutservice      trace.from_frontendservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
11     emailservice      trace.from_checkoutservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
12  checkoutservice  trace.to_productcatalogservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
13  checkoutservice        trace.to_currencyservice.error_rate         0.0             24  2024-01-19 00:07:00+00:00                0.0             NaN
14     emailservice                  trace.from_root.row_count        24.0             23  2024-01-19 00:07:00+00:00               24.0        1.000000
15  checkoutservice                  trace.from_root.row_count       135.0              3  2024-01-19 00:07:00+00:00              137.0        1.014815
16     emailservice       trace.from_checkoutservice.row_count        32.0              3  2024-01-19 00:11:00+00:00               33.0        1.031250
17  checkoutservice          trace.to_paymentservice.row_count        32.0              3  2024-01-19 00:11:00+00:00               33.0        1.031250
18  checkoutservice       trace.from_frontendservice.row_count        32.0              3  2024-01-19 00:11:00+00:00               33.0        1.031250
19  checkoutservice         trace.to_currencyservice.row_count       111.0              3  2024-01-19 00:07:00+00:00              113.0        1.018018, 1260)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse df_log and df_err if available; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv')

# Ensure timestamps parsed as UTC datetimes
for df in (df_log, df_err):
    if 'timestamp' in df.columns and not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P5 per (cmdb_id, log_name) using the full log series
log_p95 = df_log.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_p95'})
log_p05 = df_log.groupby(['cmdb_id', 'log_name'])['value'].quantile(0.05).reset_index().rename(columns={'value': 'global_p5'})

log_thresholds = log_p95.merge(log_p05, on=['cmdb_id', 'log_name'], how='outer')

# 2) Define incident window and candidate services
start = pd.Timestamp('2024-01-19 00:03:59', tz='UTC')
end = pd.Timestamp('2024-01-19 00:33:59', tz='UTC')
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# 3) Filter log rows to incident window and candidate services
df_log_window = df_log[
    (df_log['timestamp'] >= start) &
    (df_log['timestamp'] <= end) &
    (df_log['cmdb_id'].isin(candidates))
].copy()

total_log_rows_in_window = len(df_log_window)

# 4) Merge windowed logs with thresholds (computed from full series)
df_log_window = df_log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 5) High anomalies: value >= global_p95 (exclude rows without threshold)
high_mask = df_log_window['global_p95'].notna() & (df_log_window['value'] >= df_log_window['global_p95'])
df_high_anoms = df_log_window[high_mask].copy()

if not df_high_anoms.empty:
    high_agg = (
        df_high_anoms
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(
            global_p95=('global_p95', 'first'),
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            max_anomaly_value=('value', 'max')
        )
    )
    high_agg['severity_ratio'] = np.where(
        high_agg['global_p95'] == 0,
        np.nan,
        high_agg['max_anomaly_value'] / high_agg['global_p95']
    )
    # rounding
    high_agg['global_p95'] = high_agg['global_p95'].round(6)
    high_agg['max_anomaly_value'] = high_agg['max_anomaly_value'].round(6)
    high_agg['severity_ratio'] = pd.Series(high_agg['severity_ratio']).round(6)
    # annotate type and standardize columns
    high_summary = high_agg.rename(columns={
        'global_p95': 'threshold_value',
        'max_anomaly_value': 'extremum_value'
    })
    high_summary.insert(2, 'threshold_name', 'global_p95')
    high_summary.insert(6, 'ratio_name', 'severity_ratio')
    high_summary = high_summary[['cmdb_id','log_name','threshold_name','threshold_value','anomaly_count','earliest_anomaly_timestamp','extremum_value','ratio_name','severity_ratio']]
else:
    high_summary = pd.DataFrame(columns=['cmdb_id','log_name','threshold_name','threshold_value','anomaly_count','earliest_anomaly_timestamp','extremum_value','ratio_name','severity_ratio'])

# 6) Low anomalies: value <= global_p5 (exclude rows without threshold)
low_mask = df_log_window['global_p5'].notna() & (df_log_window['value'] <= df_log_window['global_p5'])
df_low_anoms = df_log_window[low_mask].copy()

if not df_low_anoms.empty:
    low_agg = (
        df_low_anoms
        .groupby(['cmdb_id', 'log_name'], as_index=False)
        .agg(
            global_p5=('global_p5', 'first'),
            anomaly_count=('value', 'size'),
            earliest_anomaly_timestamp=('timestamp', 'min'),
            min_anomaly_value=('value', 'min')
        )
    )
    low_agg['drop_ratio'] = np.where(
        low_agg['min_anomaly_value'] == 0,
        np.nan,
        low_agg['global_p5'] / low_agg['min_anomaly_value']
    )
    # rounding
    low_agg['global_p5'] = low_agg['global_p5'].round(6)
    low_agg['min_anomaly_value'] = low_agg['min_anomaly_value'].round(6)
    low_agg['drop_ratio'] = pd.Series(low_agg['drop_ratio']).round(6)
    # annotate type and standardize columns
    low_summary = low_agg.rename(columns={
        'global_p5': 'threshold_value',
        'min_anomaly_value': 'extremum_value'
    })
    low_summary.insert(2, 'threshold_name', 'global_p5')
    low_summary.insert(6, 'ratio_name', 'drop_ratio')
    low_summary = low_summary[['cmdb_id','log_name','threshold_name','threshold_value','anomaly_count','earliest_anomaly_timestamp','extremum_value','ratio_name','drop_ratio']]
else:
    low_summary = pd.DataFrame(columns=['cmdb_id','log_name','threshold_name','threshold_value','anomaly_count','earliest_anomaly_timestamp','extremum_value','ratio_name','drop_ratio'])

# 7) Combine high and low summaries, sort by anomaly_count desc, limit combined top 20
combined = pd.concat([high_summary, low_summary], ignore_index=True, sort=False)
if not combined.empty:
    combined_sorted = combined.sort_values('anomaly_count', ascending=False).head(20).reset_index(drop=True)
else:
    combined_sorted = combined

# 8) error_logs subset within the incident window for candidates (timestamp, cmdb_id, message), limit top 20
err_window = df_err[
    (df_err['timestamp'] >= start) &
    (df_err['timestamp'] <= end) &
    (df_err['cmdb_id'].isin(candidates))
].copy().sort_values('timestamp').loc[:, ['timestamp','cmdb_id','message']].head(20).reset_index(drop=True)

# Final compact outputs
combined_sorted, total_log_rows_in_window, err_window

```
Out[4]:
```
Summary of log anomalies for the candidate services in the incident window (2024-01-19 00:03:59 – 00:33:59 UTC):

- Total log rows for the candidate services in the window: 196.
- error_logs subset for these services in the window: none (empty).

Key findings (compact):

- Many log.error_count groups show global P95 = 0 and have anomaly_count = 24–25 (values = 0). These are not actionable (threshold = 0, no real error spike).

- Meaningful high anomalies (value >= global_p95):
  - emailservice — log.row_count: anomaly_count = 3, earliest = 2024-01-19 00:11:00 UTC, max = 33, global_p95 = 32, severity ≈ 1.031
  - checkoutservice — log.row_count: anomaly_count = 3, earliest = 2024-01-19 00:11:00 UTC, max = 99, global_p95 = 96, severity ≈ 1.031
  - currencyservice — log.row_count: anomaly_count = 2, earliest = 2024-01-19 00:28:00 UTC, max = 2554, global_p95 ≈ 2522.20, severity ≈ 1.013
  - recommendationservice — log.row_count: anomaly_count = 2, earliest = 2024-01-19 00:07:00 UTC, max = 611, global_p95 ≈ 601.60, severity ≈ 1.016

- Meaningful low anomalies (value <= global_p5):
  - emailservice — log.row_count: anomaly_count = 3, earliest = 2024-01-19 00:13:00 UTC, min = 20, global_p5 = 22, drop_ratio = 1.10
  - checkoutservice — log.row_count: anomaly_count = 2, earliest = 2024-01-19 00:16:00 UTC, min = 60, global_p5 ≈ 66.15, drop_ratio ≈ 1.1025
  - currencyservice — log.row_count: anomaly_count = 2, earliest = 2024-01-19 00:06:00 UTC, min = 26, global_p5 ≈ 2193.20, drop_ratio ≈ 84.35 (very large because min is tiny relative to p5)
  - recommendationservice — log.row_count: anomaly_count = 2, earliest = 2024-01-19 00:06:00 UTC, min = 5, global_p5 ≈ 526.60, drop_ratio ≈ 105.32 (very large due to tiny min)

Interpretation and cautions:
- Most high-severity ratios are small (~1.01–1.03): short, mild increases in log.row_count for emailservice, checkoutservice, currencyservice, recommendationservice.
- Very large drop_ratio values for currencyservice and recommendationservice are driven by a few very small min values (min = 26 or 5) compared to much larger typical values — this produces large ratio numbers but may reflect intermittent low counts or sampling/ingestion irregularities rather than a broad outage.
- log.error_count entries at zero with global thresholds of zero are not useful for signalling problems here.

Bottom line: There are modest, short-lived increases in log.row_count for several services (severity ~1.01–1.03). A few extreme low readings produce large drop ratios but should be examined carefully (likely isolated low-count datapoints or instrumentation variance). There are no error_logs in the window to corroborate widespread failures.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name threshold_name  threshold_value  anomaly_count earliest_anomaly_timestamp  extremum_value      ratio_name  severity_ratio  drop_ratio
0         currencyservice  log.error_count     global_p95             0.00             25  2024-01-19 00:06:00+00:00               0  severity_ratio             NaN         NaN
1   recommendationservice  log.error_count     global_p95             0.00             25  2024-01-19 00:06:00+00:00               0  severity_ratio             NaN         NaN
2   recommendationservice  log.error_count      global_p5             0.00             25  2024-01-19 00:06:00+00:00               0      drop_ratio             NaN         NaN
3         currencyservice  log.error_count      global_p5             0.00             25  2024-01-19 00:06:00+00:00               0      drop_ratio             NaN         NaN
4         checkoutservice  log.error_count      global_p5             0.00             24  2024-01-19 00:07:00+00:00               0      drop_ratio             NaN         NaN
5            emailservice  log.error_count      global_p5             0.00             24  2024-01-19 00:07:00+00:00               0      drop_ratio             NaN         NaN
6            emailservice  log.error_count     global_p95             0.00             24  2024-01-19 00:07:00+00:00               0  severity_ratio             NaN         NaN
7         checkoutservice  log.error_count     global_p95             0.00             24  2024-01-19 00:07:00+00:00               0  severity_ratio             NaN         NaN
8            emailservice    log.row_count      global_p5            22.00              3  2024-01-19 00:13:00+00:00              20      drop_ratio             NaN    1.100000
9            emailservice    log.row_count     global_p95            32.00              3  2024-01-19 00:11:00+00:00              33  severity_ratio        1.031250         NaN
10        checkoutservice    log.row_count     global_p95            96.00              3  2024-01-19 00:11:00+00:00              99  severity_ratio        1.031250         NaN
11        currencyservice    log.row_count     global_p95          2522.20              2  2024-01-19 00:28:00+00:00            2554  severity_ratio        1.012608         NaN
12  recommendationservice    log.row_count     global_p95           601.60              2  2024-01-19 00:07:00+00:00             611  severity_ratio        1.015625         NaN
13        checkoutservice    log.row_count      global_p5            66.15              2  2024-01-19 00:16:00+00:00              60      drop_ratio             NaN    1.102500
14        currencyservice    log.row_count      global_p5          2193.20              2  2024-01-19 00:06:00+00:00              26      drop_ratio             NaN   84.353846
15  recommendationservice    log.row_count      global_p5           526.60              2  2024-01-19 00:06:00+00:00               5      drop_ratio             NaN  105.320000, 196, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```